# Data Check — Verificación de calidad de datos

> Notebook de investigación que compara datos **raw** (CSV crudos) contra **processed** (Parquet post-ETL) para detectar pérdidas de información, inconsistencias en merges, y problemas de filtrado que puedan estar eliminando casos positivos legítimos.

**Fuentes investigadas:** maestros, inspecciones, consumos, y dataset final.

In [12]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

# ============================================================
# ⚠️  CONFIGURACIÓN: cambiá esta ruta al path de tu proyecto
# ============================================================
PROJECT_PATH = Path("<PATH_DEL_PROYECTO>")
# Ejemplo: Path("/home/usuario/mi_proyecto") o Path("../.proyects/mi_empresa")
RAW_PATH = PROJECT_PATH / "data/raw/v0"
PROC_PATH = PROJECT_PATH / "data/processed/v0"

# Nombre del archivo de dataset final (producto del ETL)
DATASET_FILE = PROC_PATH / "<NOMBRE_DEL_DATASET>.parquet"
# Ejemplo: PROC_PATH / "dataset.parquet"

### Maestro — Datos maestros de clientes

El archivo de **maestros** contiene los datos catastrales de cada cliente: identificador, nombre, dirección, coordenadas geográficas, y variables categóricas como actividad económica, tipo de tarifa, nivel de tensión, material de instalación y zona geográfica.

Comparamos la versión **raw** (CSV crudo) contra la versión **processed** (Parquet generado por el ETL) para verificar que la transformación no introdujo pérdidas ni cambios inesperados.

In [14]:
maestro_csv = pd.read_csv(RAW_PATH/"maestros/data_concatenated.csv", sep=",", encoding="utf-8-sig")
maestro_csv["CLIENTE"] = maestro_csv["CLIENTE"].astype(str).str.strip()
print(f"Raw  (CSV):     {maestro_csv.shape}")

maestro_pq = pd.read_parquet(PROC_PATH/"maestros.parquet")
print(f"Proc (Parquet): {maestro_pq.shape}")


ParserError: Error tokenizing data. C error: Expected 1 fields in line 99, saw 3


In [ ]:
# --- Comparación raw vs processed ---
print("=== Maestros: Raw vs Processed ===\n")

# 1. Misma cantidad de filas?
print(f"Raw filas:  {len(maestro_csv):,}")
print(f"Proc filas: {len(maestro_pq):,}")
print(f"Diferencia: {len(maestro_csv) - len(maestro_pq):,} filas\n")

# 2. Mismos clientes?
raw_clientes = set(maestro_csv["CLIENTE"])
proc_clientes = set(maestro_pq["cliente"])
print(f"Clientes en raw:  {len(raw_clientes):,}")
print(f"Clientes en proc: {len(proc_clientes):,}")
print(f"Solo en raw:  {len(raw_clientes - proc_clientes):,}")
print(f"Solo en proc: {len(proc_clientes - raw_clientes):,}\n")

# 3. Duplicados?
dup_raw = maestro_csv["CLIENTE"].duplicated().sum()
dup_proc = maestro_pq["cliente"].duplicated().sum()
print(f"Clientes duplicados — raw: {dup_raw}, proc: {dup_proc}\n")

# 4. Columnas
print(f"Columnas raw:  {list(maestro_csv.columns)}")
print(f"Columnas proc: {list(maestro_pq.columns)}")


### Inspecciones — Resultados de fiscalización en campo

Las **inspecciones** son el ground truth del problema: cada fila es una visita de un inspector a un cliente, con el resultado (`TARGET`/`target`): 1 = fraudulento, 0 = normal.

Comparamos raw vs processed para asegurar que la cantidad de positivos se preserva correctamente.

In [ ]:
files = Path(RAW_PATH/"inspecciones").glob("data*.csv")
inspec_csv = pd.concat((pd.read_csv(f, sep=",", encoding="utf-8-sig") for f in files), ignore_index=True)
inspec_csv["CLIENTE"] = inspec_csv["CLIENTE"].astype(str).str.strip()
print(inspec_csv.shape)

inspec_pq = pd.read_parquet(PROC_PATH/"inspecciones")
print(inspec_pq.shape)

In [ ]:
inspec_csv.head()

In [ ]:
inspec_pq.sort_values(["cliente"]).head()

In [ ]:
inspec_csv.TARGET.value_counts(), inspec_pq.target.value_counts()

In [ ]:
# --- Comparación raw vs processed ---
print("=== Inspecciones: Raw vs Processed ===\n")

# 1. Misma cantidad de filas?
print(f"Raw filas:  {len(inspec_csv):,}")
print(f"Proc filas: {len(inspec_pq):,}")
print(f"Diferencia: {len(inspec_csv) - len(inspec_pq):,} filas\n")

# 2. Distribución de target (¿se preserva?)
print("--- Distribución TARGET/target ---")
print("Raw:")
print(inspec_csv["TARGET"].value_counts())
print(f"  % positivos: {100 * inspec_csv['TARGET'].mean():.2f}%")
print()
print("Proc:")
print(inspec_pq["target"].value_counts())
print(f"  % positivos: {100 * inspec_pq['target'].mean():.2f}%")
print()

# 3. Mismos clientes?
raw_ins_cli = set(inspec_csv["CLIENTE"])
proc_ins_cli = set(inspec_pq["cliente"])
print(f"Clientes únicos — raw: {len(raw_ins_cli):,}, proc: {len(proc_ins_cli):,}")
print(f"Solo en raw:  {len(raw_ins_cli - proc_ins_cli):,}")
print(f"Solo en proc: {len(proc_ins_cli - raw_ins_cli):,}\n")

# 4. ¿Hay clientes con múltiples inspecciones?
multi_raw = inspec_csv["CLIENTE"].value_counts()
multi_proc = inspec_pq["cliente"].value_counts()
print(f"Clientes con >1 inspección — raw: {(multi_raw > 1).sum():,}, proc: {(multi_proc > 1).sum():,}")
print(f"Máx inspecciones por cliente — raw: {multi_raw.max()}, proc: {multi_proc.max()}")


### Consumos — Series temporales de consumo mensual

Los **consumos** son la serie histórica de lecturas mensuales por cliente. Cada fila es un par `(cliente, periodo, consumo)`. Estos datos son la materia prima para:

- Derivar features estadísticas, espectrales y temporales
- Detectar patrones anómalos (caídas bruscas, consumo constante, estacionalidad)

Comparamos raw vs processed para verificar consistencia en cantidad de registros.

In [ ]:
files = Path(RAW_PATH/"consumos").glob("data*.csv")
consumo_csv = pd.concat((pd.read_csv(f, sep=",", encoding="utf-8-sig") for f in files), ignore_index=True)
consumo_csv["CLIENTE"] = consumo_csv["CLIENTE"].astype(str).str.strip()
print(f"Raw  (CSV):     {consumo_csv.shape}")

consumo_pq = pd.read_parquet(PROC_PATH/"consumos")
print(f"Proc (Parquet): {consumo_pq.shape}")


In [ ]:
# --- Comparación raw vs processed ---
print("=== Consumos: Raw vs Processed ===\n")

# 1. Misma cantidad de registros?
print(f"Raw filas:  {len(consumo_csv):,}")
print(f"Proc filas: {len(consumo_pq):,}")
print(f"Diferencia: {len(consumo_csv) - len(consumo_pq):,} filas\n")

# 2. Rango de períodos
print(f"Raw  — período min: {consumo_csv['PERIODO'].min()}, max: {consumo_csv['PERIODO'].max()}")
print(f"Proc — período min: {consumo_pq['periodo'].min()}, max: {consumo_pq['periodo'].max()}\n")

# 3. Mismos clientes?
raw_con_cli = set(consumo_csv["CLIENTE"])
proc_con_cli = set(consumo_pq["cliente"])
print(f"Clientes únicos — raw: {len(raw_con_cli):,}, proc: {len(proc_con_cli):,}")
print(f"Solo en raw:  {len(raw_con_cli - proc_con_cli):,}")
print(f"Solo en proc: {len(proc_con_cli - raw_con_cli):,}\n")

# 4. Valores nulos en consumo
consumo_raw_nulls = consumo_csv["CONSUMO"].isna().sum()
consumo_proc_nulls = consumo_pq["consumo"].isna().sum()
print(f"Consumos nulos — raw: {consumo_raw_nulls:,} ({100*consumo_raw_nulls/len(consumo_csv):.2f}%), "
      f"proc: {consumo_proc_nulls:,} ({100*consumo_proc_nulls/len(consumo_pq):.2f}%)")
print(f"Consumos = 0   — raw: {(consumo_csv['CONSUMO'] == 0).sum():,}, "
      f"proc: {(consumo_pq['consumo'] == 0).sum():,}")


### Dataset — Producto final del ETL

El **dataset** es el resultado de mergear maestros, inspecciones y consumos en formato *wide*: una fila por cliente, con 12 columnas de consumo mensual (`12_anterior` … `1_anterior`) más las variables categóricas y el target.

Este es el punto de entrada al pipeline de entrenamiento. Cargamos el dataset procesado para inspeccionar su estructura y verificar la cantidad de positivos.

In [ ]:
df = pd.read_parquet(DATASET_FILE)

In [ ]:
# --- Inspección del dataset procesado ---
print("=== Dataset procesado ===\n")
print(f"Shape: {df.shape}")
print(f"\nColumnas ({len(df.columns)}):")
print(list(df.columns))
print(f"\nTipos de dato:")
print(df.dtypes.value_counts())
print(f"\nValores nulos por columna (top 10):")
nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls[nulls > 0].head(10))
print(f"\nTarget: {df['target'].value_counts().to_dict()}")
print(f"% positivos: {100 * df['target'].mean():.2f}%")


In [ ]:
# --- Consistencia con fuentes ---
print("=== Consistencia cruzada ===\n")

# ¿Los clientes del dataset están en maestros, inspecciones y consumos?
ds_clientes = set(df["cliente"])
print(f"Clientes en dataset:     {len(ds_clientes):,}")
print(f"  En maestros:          {len(ds_clientes & proc_clientes):,} ({100*len(ds_clientes & proc_clientes)/len(ds_clientes):.1f}%)")
print(f"  En inspecciones:      {len(ds_clientes & proc_ins_cli):,} ({100*len(ds_clientes & proc_ins_cli)/len(ds_clientes):.1f}%)")
print(f"  En consumos:          {len(ds_clientes & proc_con_cli):,} ({100*len(ds_clientes & proc_con_cli)/len(ds_clientes):.1f}%)")
print(f"  En las 3 fuentes:     {len(ds_clientes & proc_clientes & proc_ins_cli & proc_con_cli):,}")
print(f"\nClientes en dataset que NO están en maestros: {len(ds_clientes - proc_clientes):,}")
print(f"Clientes en dataset que NO están en consumos:  {len(ds_clientes - proc_con_cli):,}")
print(f"Clientes en dataset que NO están en inspecciones: {len(ds_clientes - proc_ins_cli):,}")


### Inspecciones positivas — ¿Cuántos fraudulentos hay?

Revisamos cuántos casos positivos (target = 1) hay en las inspecciones raw vs el dataset procesado. Si hay diferencias grandes, el ETL podría estar filtrando fraudulentos legítimos.

También inspeccionamos un cliente positivo de ejemplo para entender cómo se ven sus datos en cada etapa del pipeline.

In [ ]:
inspec_csv[inspec_csv.TARGET == 1].shape

In [ ]:
inspec_pq[inspec_pq.target == 1].shape

In [ ]:
inspec_pq[inspec_pq.target == 1].sample(5)

In [ ]:
cliente = "28865864"
inspec_csv[inspec_csv.CLIENTE == cliente]

In [ ]:
inspec_pq[inspec_pq.cliente == cliente]

In [ ]:
consumo_csv[consumo_csv.CLIENTE == cliente].sort_values(['PERIODO'])

In [ ]:
consumo_pq[consumo_pq.cliente == cliente].sort_values(['periodo'])

In [ ]:
df[df.cliente == cliente]

In [ ]:
df.target.value_counts()

### Análisis: ¿por qué solo 172 positivos en el dataset procesado?

Las inspecciones crudas contienen **3,963** casos positivos, pero el dataset procesado solo retiene **172**. Esto representa una pérdida del **95.7%** de los positivos, lo cual es alarmante.

**Hipótesis investigadas:**

1. **Filtro `min_num_measures_not_zero >= 1`**: el ETL descarta clientes que no tienen al menos un período con consumo > 0. Si un cliente fraudulento tiene todos sus consumos en cero, este filtro lo elimina — pero consumo todo en cero también puede ser una señal de fraude (medidor puenteado o dado de baja).

2. **Cobertura de consumos**: ¿los positivos eliminados directamente no tienen datos en la tabla de consumos? Si el inspector visitó a un cliente que no está en el sistema de medición, no hay serie temporal que analizar.

3. **Match de identificadores**: ¿hay diferencias en el formato del `CLIENTE` entre inspecciones, maestros y consumos que rompan el merge?

**Lo que hacemos a continuación:**
- Reconstruir el dataset sin el filtro `min_num_measures_not_zero` para ver cuántos positivos se recuperan
- Analizar la distribución de `num_measures` (cantidad de períodos con dato) para los eliminados
- Verificar si los positivos eliminados existen en la tabla de consumos
- Inspeccionar el merge de identificadores entre las tres fuentes

In [ ]:
# Reconstruir el dataset SIN el filtro min_num_measures_not_zero
# para ver qué pasa con los positivos eliminados

maestros = pd.read_parquet(PROC_PATH / "maestros.parquet")
consumos = pd.read_parquet(PROC_PATH / "consumos")
inspecciones = pd.read_parquet(PROC_PATH / "inspecciones")

consumption_cols = [f"{i}_anterior" for i in range(12, 0, -1)]

# --- Pivot consumos (misma lógica que DatasetBuilderETL) ---
joined = inspecciones[["cliente", "periodo"]].merge(
    consumos, on="cliente", suffixes=("_insp", "_cons")
)

insp_dt = pd.to_datetime(joined["periodo_insp"], format="%Y%m")
cons_dt = pd.to_datetime(joined["periodo_cons"], format="%Y%m")
joined["rank"] = ((insp_dt.dt.year - cons_dt.dt.year) * 12 + (insp_dt.dt.month - cons_dt.dt.month)).astype(int)
joined = joined[(joined["rank"] >= 1) & (joined["rank"] <= 12)]
joined["col_name"] = joined["rank"].astype(str) + "_anterior"
joined = joined.drop_duplicates(subset=["cliente", "periodo_insp", "col_name"], keep="first")

consumos_wide = (
    joined.pivot(index=["cliente", "periodo_insp"], columns="col_name", values="consumo")
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={"periodo_insp": "periodo"})
)

# Left join para mantener TODAS las inspecciones
all_insp = inspecciones[["cliente", "periodo"]].drop_duplicates()
consumos_wide = all_insp.merge(consumos_wide, on=["cliente", "periodo"], how="left")
consumos_wide = consumos_wide.reindex(columns=["cliente", "periodo"] + consumption_cols, fill_value=float("nan"))

# Join con inspecciones y maestros
result = inspecciones.merge(consumos_wide, on=["cliente", "periodo"], how="inner")
result = result.merge(maestros, on="cliente", how="inner")

# Calcular métricas de calidad
result["num_measures"] = result[consumption_cols].notnull().sum(axis=1)
result["num_measures_not_zero"] = (result[consumption_cols] > 0).sum(axis=1)
# NUEVA: ¿tiene algún dato de consumo (no NaN)?
result["has_any_data"] = result[consumption_cols].notnull().any(axis=1)

# Filtro geo (mismo que el ETL)
result = result[
    result["latitude"].notna() & result["longitude"].notna()
    & (result["latitude"] != 0.0) & (result["longitude"] != 0.0)
]

print(f"Total después de geo filter: {len(result)} ({result['target'].sum()} positivos)")
print(f"\n--- Desglose de positivos ---")

positivos = result[result["target"] == 1]
print(f"Total positivos: {len(positivos)}")
print(f"  Con al menos 1 medida > 0 (sobreviven filtro): {(positivos['num_measures_not_zero'] >= 1).sum()}")
print(f"  Con 0 medidas > 0 (eliminados por filtro):     {(positivos['num_measures_not_zero'] == 0).sum()}")
print(f"\nDe los eliminados:")
eliminados = positivos[positivos["num_measures_not_zero"] == 0]
print(f"  Sin NINGÚN dato de consumo (todo NaN):    {(~eliminados['has_any_data']).sum()}")
print(f"  Con datos pero todos en cero:             {(eliminados['has_any_data']).sum()}")

In [ ]:
# Detalle: distribución de num_measures (cuántos periodos tienen dato) para los eliminados
print("--- Distribución de num_measures en positivos eliminados (num_measures_not_zero == 0) ---")
print(eliminados["num_measures"].value_counts().sort_index())
print(f"\n--- Distribución de num_measures en positivos que sobreviven ---")
sobreviven = positivos[positivos["num_measures_not_zero"] >= 1]
print(sobreviven["num_measures"].value_counts().sort_index())

# Comparar negativos
negativos = result[result["target"] == 0]
print(f"\n--- Negativos ---")
print(f"Total negativos: {len(negativos)}")
print(f"  Con 0 medidas > 0: {(negativos['num_measures_not_zero'] == 0).sum()}")
print(f"  Sin ningún dato:   {(~negativos[negativos['num_measures_not_zero'] == 0]['has_any_data']).sum()}")

In [ ]:
# ¿Los positivos eliminados existen en consumos?
# Verificar si sus clientes tienen ALGÚN registro de consumo (cualquier período)
clientes_eliminados = set(eliminados["cliente"].unique())
clientes_en_consumos = set(consumos["cliente"].unique())

en_consumos = clientes_eliminados & clientes_en_consumos
no_en_consumos = clientes_eliminados - clientes_en_consumos

print(f"Clientes positivos eliminados (únicos): {len(clientes_eliminados)}")
print(f"  Existen en tabla consumos:     {len(en_consumos)}")
print(f"  NO existen en tabla consumos:  {len(no_en_consumos)}")
print(f"\nInspecciones positivas eliminadas: {len(eliminados)}")
print(f"  De clientes que SÍ están en consumos: {eliminados['cliente'].isin(en_consumos).sum()}")
print(f"  De clientes que NO están en consumos: {eliminados['cliente'].isin(no_en_consumos).sum()}")

In [ ]:
inspecciones.head()

In [ ]:
consumos.head()

---
## Resumen de hallazgos

Completá esta sección después de ejecutar el notebook:

| Check | Raw | Processed | OK? |
|-------|-----|-----------|-----|
| Maestros: filas | | | |
| Maestros: duplicados | | | |
| Inspecciones: filas | | | |
| Inspecciones: % positivos | | | |
| Inspecciones: clientes únicos | | | |
| Consumos: filas | | | |
| Consumos: clientes únicos | | | |
| Consumos: período min-max | | | |
| Dataset: filas | — | | |
| Dataset: % positivos | — | | |
| Cross-check: clientes en las 3 fuentes | — | | |
| Positivos retenidos vs originales | 3,963 | 172 | ❌ 95.7% perdidos |

**Conclusión principal:** el filtro `min_num_measures_not_zero >= 1` es el principal responsable de la pérdida de positivos. Revisar si corresponde relajar este filtro o tratar los consumos todo-cero como una señal en lugar de descartarlos.

**Próximo paso:** `02_outlier_analysis.ipynb` para analizar outliers en el dataset procesado.